# Trader Performance vs Market Sentiment Analysis

**Objective**: Analyze the relationship between market sentiment (Fear & Greed Index) and historical trader behavior on Hyperliquid.

## Setup & Libraries
Let's import the required scientific stack libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

sns.set_theme(style="whitegrid")
print("Libraries successfully imported!")

## Part A — Data Preparation & Cleaning
We will load both datasets, check row/column counts, investigate missing values, clean the timestamps, and align both datasets by date.

In [ ]:
# Load raw CSV files
sentiment = pd.read_csv('data/bitcoin_sentiment.csv')
trader = pd.read_csv('data/historical_trader_data.csv')

print(f"Bitcoin Sentiment: {sentiment.shape[0]} rows, {sentiment.shape[1]} columns")
print(f"Trader Data: {trader.shape[0]} rows, {trader.shape[1]} columns")

# Missing values
print("\nMissing values in Sentiment Data:\n", sentiment.isnull().sum())
print("\nMissing values in Trader Data:\n", trader.isnull().sum())

# Duplicate check
print(f"\nDuplicate rows: Sentiment ({sentiment.duplicated().sum()}), Trader ({trader.duplicated().sum()})")

### Aligning Timestamps
The raw `Timestamp` field in the trader data is low precision due to floating-point truncation (stored in scientific notation in the CSV). We use the string representation `Timestamp IST` parsed with `format='%d-%m-%Y %H:%M'` to retrieve precise datetimes and extract the matching `date` (at daily level).

In [ ]:
# Parse dates
sentiment['date'] = pd.to_datetime(sentiment['date'])
trader['datetime'] = pd.to_datetime(trader['Timestamp IST'], format='%d-%m-%Y %H:%M')
trader['date'] = trader['datetime'].dt.normalize()

# Categorize sentiment class: 'Fear' vs 'Greed' vs 'Neutral'
def categorize_sent(classification):
    if 'Fear' in classification:
        return 'Fear'
    elif 'Greed' in classification:
        return 'Greed'
    else:
        return 'Neutral'

sentiment['sentiment_category'] = sentiment['classification'].apply(categorize_sent)

# Merge the datasets
merged = pd.merge(trader, sentiment[['date', 'value', 'classification', 'sentiment_category']], on='date', how='inner')
print(f"Aligned merged dataset shape: {merged.shape}")

### Computing Key Metrics
We calculate key metrics per account-date (daily level) to capture changes in trader behavior. 
We define two distinct, mutually exclusive long/short metrics to avoid statistical overlaps:
1. **Taker Buy Ratio**: Proportion of trades executed on the BUY side (representing short-covering or long-opening market pressure).
2. **Long Opening Ratio**: Proportion of new position opening trades that are Long (representing directional position building bias).

In [ ]:
merged['is_win'] = merged['Closed PnL'] > 0
merged['is_trade'] = merged['Closed PnL'] != 0
merged['is_buy'] = merged['Side'] == 'BUY'
merged['is_sell'] = merged['Side'] == 'SELL'
merged['is_open_long'] = merged['Direction'] == 'Open Long'
merged['is_open_short'] = merged['Direction'] == 'Open Short'

daily = merged.groupby(['Account', 'date']).agg(
    daily_pnl=('Closed PnL', 'sum'),
    total_trades=('Trade ID', 'count'),
    closed_trades=('is_trade', 'sum'),
    winning_trades=('is_win', 'sum'),
    avg_size_usd=('Size USD', 'mean'),
    cross_margin_trades=('Crossed', 'sum'),
    buy_trades=('is_buy', 'sum'),
    sell_trades=('is_sell', 'sum'),
    open_long_trades=('is_open_long', 'sum'),
    open_short_trades=('is_open_short', 'sum'),
    sentiment_val=('value', 'first'),
    sentiment_cat=('sentiment_category', 'first')
).reset_index()

daily['win_rate'] = np.where(daily['closed_trades'] > 0, daily['winning_trades'] / daily['closed_trades'], 0.0)
daily['cross_margin_ratio'] = daily['cross_margin_trades'] / daily['total_trades']
daily['buy_ratio'] = daily['buy_trades'] / daily['total_trades']

daily['long_open_ratio'] = np.where(
    (daily['open_long_trades'] + daily['open_short_trades']) > 0,
    daily['open_long_trades'] / (daily['open_long_trades'] + daily['open_short_trades']),
    0.5
)

print(daily.head(3))

## Part B — Analysis & Visualizations
### Q1: Performance Diff (Fear vs Greed)
Let's compare PnL, win rate, and tail drawdowns (5th percentile PnL) between Fear and Greed days.

In [ ]:
fg_daily = daily[daily['sentiment_cat'].isin(['Fear', 'Greed'])]

perf = fg_daily.groupby('sentiment_cat').agg(
    avg_daily_pnl=('daily_pnl', 'mean'),
    median_daily_pnl=('daily_pnl', 'median'),
    avg_win_rate=('win_rate', 'mean'),
    drawdown_proxy_p5=('daily_pnl', lambda x: x.quantile(0.05))
)
print(perf)

### Visualizing the Risk-Return Tradeoff
Let's build a barplot comparing the average return vs tail drawdown (5th percentile PnL) across sentiment environments.

In [ ]:
comparison_df = pd.DataFrame({
    'Average Return': fg_daily.groupby('sentiment_cat')['daily_pnl'].mean(),
    'Tail Drawdown (5th Pct)': fg_daily.groupby('sentiment_cat')['daily_pnl'].quantile(0.05)
}).reset_index()

melted = pd.melt(comparison_df, id_vars='sentiment_cat', value_vars=['Average Return', 'Tail Drawdown (5th Pct)'])

plt.figure(figsize=(9, 5.5))
sns.barplot(x='sentiment_cat', y='value', hue='variable', data=melted, palette='muted')
plt.title('Risk-Return Tradeoff: Fear vs Greed Days')
plt.xlabel('Market Sentiment')
plt.ylabel('Value (USD)')
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

### Q2: Behavioral Shifts (Taker Buy, Long Opening, and Cross Margin Ratios)
Do traders shift positions or exposure when sentiment changes? Let's check using box plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

sns.boxplot(x='sentiment_cat', y='buy_ratio', data=fg_daily, ax=axes[0], palette='coolwarm', hue='sentiment_cat', legend=False)
axes[0].set_title('Taker Buy Ratio Shifts')
axes[0].set_xlabel('Market Sentiment')
axes[0].set_ylabel('Buy Trades / Total Trades')

sns.boxplot(x='sentiment_cat', y='long_open_ratio', data=fg_daily, ax=axes[1], palette='coolwarm', hue='sentiment_cat', legend=False)
axes[1].set_title('Long Opening Ratio Shifts (New Positions)')
axes[1].set_xlabel('Market Sentiment')
axes[1].set_ylabel('Open Longs / Total Opens')

sns.boxplot(x='sentiment_cat', y='cross_margin_ratio', data=fg_daily, ax=axes[2], palette='coolwarm', hue='sentiment_cat', legend=False)
axes[2].set_title('Cross Margin Exposure Shifts')
axes[2].set_xlabel('Market Sentiment')
axes[2].set_ylabel('Cross Margin Trades / Total Trades')

plt.suptitle('Trader Behavioral Modifications Under Sentiment Shift')
plt.tight_layout()
plt.show()

### Q3: Behavioral Segmentation
We segment traders by their margin preference (Cross vs Isolated Margin) and trading frequency. Additionally, we run K-Means to identify behavioral archetypes.

In [ ]:
# Aggregating trader profiles
account_df = merged.groupby('Account').agg(
    total_pnl=('Closed PnL', 'sum'),
    total_trades=('Trade ID', 'count'),
    closed_trades=('is_trade', 'sum'),
    winning_trades=('is_win', 'sum'),
    avg_trade_size=('Size USD', 'mean'),
    cross_margin_ratio=('Crossed', 'mean'),
    buy_ratio=('is_buy', 'mean'),
    open_longs=('is_open_long', 'sum'),
    open_shorts=('is_open_short', 'sum')
).reset_index()

account_df['win_rate'] = np.where(account_df['closed_trades'] > 0, account_df['winning_trades'] / account_df['closed_trades'], 0.0)
account_df['long_open_ratio'] = np.where(
    (account_df['open_longs'] + account_df['open_shorts']) > 0,
    account_df['open_longs'] / (account_df['open_longs'] + account_df['open_shorts']),
    0.5
)

# Segments
account_df['margin_segment'] = pd.cut(account_df['cross_margin_ratio'], bins=[-0.01, 0.3, 0.7, 1.01], labels=['Isolated Margin Heavy', 'Mixed', 'Cross Margin Heavy'])
account_df['freq_segment'] = pd.cut(account_df['total_trades'], bins=[0, 100, 1000, np.inf], labels=['Infrequent', 'Moderate', 'Frequent'])
account_df['profit_segment'] = np.where((account_df['total_pnl'] > 0) & (account_df['win_rate'] > 0.5), 'Consistent Winner', np.where(account_df['total_pnl'] < 0, 'Losing/Inconsistent', 'Neutral/Slight Winner'))

print(account_df['margin_segment'].value_counts())
print(account_df['freq_segment'].value_counts())
print(account_df['profit_segment'].value_counts())

### K-Means Clustering
Let's scale features and identify 3 K-Means clusters.

In [ ]:
features = ['total_trades', 'avg_trade_size', 'cross_margin_ratio', 'win_rate', 'long_open_ratio']
scaler = StandardScaler()
scaled = scaler.fit_transform(account_df[features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
account_df['cluster'] = kmeans.fit_predict(scaled)

profile_summary = account_df.groupby('cluster')[features + ['total_pnl']].mean()
print(profile_summary)

Let's visualize the clusters in a log-scale scatter plot.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='avg_trade_size', 
    y='total_trades', 
    hue='cluster', 
    style='profit_segment',
    size='win_rate',
    sizes=(30, 250),
    data=account_df, 
    palette='Set1',
    alpha=0.85
)
plt.title('Trader Behavioral Archetypes (K-Means Clustering)')
plt.xlabel('Average Trade Size (USD)')
plt.ylabel('Total Trades executed')
plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Bonus — Predictive Model
We predict next-day profitability (Profit vs Loss) using today's behavioral stats + sentiment levels. We use a Random Forest Classifier.

In [ ]:
daily_sorted = daily.sort_values(by=['Account', 'date'])
daily_sorted['next_day_pnl'] = daily_sorted.groupby('Account')['daily_pnl'].shift(-1)
daily_sorted['next_day_profit'] = (daily_sorted['next_day_pnl'] > 0).astype(int)
model_data = daily_sorted.dropna(subset=['next_day_pnl']).copy()

rf_features = ['daily_pnl', 'total_trades', 'win_rate', 'avg_size_usd', 'cross_margin_ratio', 'buy_ratio', 'long_open_ratio', 'sentiment_val']
X = model_data[rf_features]
y = model_data['next_day_profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=150, random_state=42, max_depth=8)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### Feature Importances
Which features are the most critical predictors of next-day trader success?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=rf_features).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis', hue=importances.index, legend=False)
plt.title('Predictive Model: Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()